# Workforce Optimization — Multi-Objective with cuOpt

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVIDIA/cuopt-examples/blob/main/workforce_optimization/workforce_optimization_multiobjective.ipynb)

The base `workforce_optimization_milp` notebook minimizes labor cost with coverage **hard-constrained** — it returns **one plan**. But that plan answers only *"cheapest way to fully staff."* A planner usually faces a **tradeoff with no fixed weighting**: *how much coverage is worth how much cost?* and *how much does fairness cost?* A single solve hides that; you get one point on a curve you can't see.

This notebook follows the `cuopt-multi-objective-exploration` skill to turn the single solve into the **whole tradeoff curve**, so the planner can see the options and choose. Two tradeoffs, both built by promoting one of the base model's hard constraints into an objective:

1. **cost vs. coverage** — relax `coverage == required` and sweep a coverage floor.
2. **cost vs. fairness** — sweep the base model's fixed `max_shifts` cap.

> **Requirements.** cuOpt needs **Linux + an NVIDIA GPU** (Colab: *Runtime → Change runtime type → GPU*).

## Environment Setup

In [ ]:
import subprocess
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True)
    print(out.stdout.strip() or "no nvidia-smi output")
except FileNotFoundError:
    print("No NVIDIA GPU detected - cuOpt cannot run. In Colab: Runtime -> Change runtime type -> GPU.")

In [ ]:
# Uncomment if cuOpt is not already installed (e.g., Google Colab):
# !pip install --upgrade --extra-index-url https://pypi.nvidia.com cuopt-cu12

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cuopt.linear_programming.problem import Problem, VType, sense, LinearExpression
from cuopt.linear_programming.solver_settings import SolverSettings
print("Imports ready")

## Problem data

Same workers, shifts, pay, and availability as the base `workforce_optimization_milp` notebook.

In [ ]:
shift_requirements = {
    "Mon1": 3, "Tue2": 2, "Wed3": 4, "Thu4": 2, "Fri5": 5, "Sat6": 3, "Sun7": 4,
    "Mon8": 2, "Tue9": 2, "Wed10": 3, "Thu11": 4, "Fri12": 5, "Sat13": 7, "Sun14": 5,
}
worker_pay = {"Amy": 10, "Bob": 12, "Cathy": 10, "Dan": 8, "Ed": 8, "Fred": 9, "Gu": 11}
availability = {
    "Amy":   ["Tue2","Wed3","Fri5","Sun7","Tue9","Wed10","Thu11","Fri12","Sat13","Sun14"],
    "Bob":   ["Mon1","Tue2","Fri5","Sat6","Mon8","Thu11","Sat13","Sun14"],
    "Cathy": ["Wed3","Thu4","Fri5","Sun7","Mon8","Tue9","Wed10","Thu11","Fri12","Sat13","Sun14"],
    "Dan":   ["Tue2","Wed3","Fri5","Sat6","Mon8","Tue9","Wed10","Thu11","Fri12","Sat13","Sun14"],
    "Ed":    ["Mon1","Tue2","Wed3","Thu4","Fri5","Sun7","Mon8","Tue9","Thu11","Sat13","Sun14"],
    "Fred":  ["Mon1","Tue2","Wed3","Sat6","Mon8","Tue9","Fri12","Sat13","Sun14"],
    "Gu":    ["Mon1","Tue2","Wed3","Fri5","Sat6","Sun7","Mon8","Tue9","Wed10","Thu11","Fri12","Sat13","Sun14"],
}
pairs = [(w, s) for w, shifts in availability.items() for s in shifts]
TOTAL_REQUIRED = sum(shift_requirements.values())
print(f"{len(worker_pay)} workers, {len(shift_requirements)} shifts, {len(pairs)} feasible (worker,shift) pairs")
print(f"Full coverage = {TOTAL_REQUIRED} staffed shifts")

## A solver helper (one model, used for every point)

Binary `x[w,s]` for each available pair; `assigned[s] ≤ required[s]` (no overstaffing, which keeps *coverage* a clean linear count). The objective is labor cost; an optional **coverage floor** is the parametric ε-constraint we'll sweep. A `time_limit` bounds every MILP solve (the skill's practical note).

In [ ]:
def solve(coverage_floor=None, maximize_coverage=False, time_limit=10.0):
    prob = Problem("workforce")
    x = {p: prob.addVariable(name=f"{p[0]}_{p[1]}", vtype=VType.INTEGER, lb=0.0, ub=1.0) for p in pairs}
    obj = LinearExpression([], [], 0.0)
    for (w, s), var in x.items():
        coef = (-1.0) if maximize_coverage else float(worker_pay[w])   # maximize coverage = minimize -sum(x)
        if coef != 0:
            obj += var * coef
    prob.setObjective(obj, sense.MINIMIZE)
    for s, req in shift_requirements.items():                          # no overstaffing
        e = LinearExpression([], [], 0.0); has = False
        for (w, s2), var in x.items():
            if s2 == s:
                e += var; has = True
        if has:
            prob.addConstraint(e <= req, name=f"cap_{s}")
    if coverage_floor is not None:                                     # epsilon-constraint
        cov = LinearExpression([], [], 0.0)
        for var in x.values():
            cov += var
        prob.addConstraint(cov >= float(coverage_floor), name="coverage_floor")
    settings = SolverSettings()
    settings.set_parameter("time_limit", float(time_limit))
    settings.set_parameter("log_to_console", False)
    prob.solve(settings)
    if prob.Status.name not in ("Optimal", "FeasibleFound"):
        return None
    sel = [(w, s) for (w, s), var in x.items() if var.getValue() > 0.5]
    return {"cost": sum(worker_pay[w] for (w, s) in sel), "coverage": len(sel), "status": prob.Status.name}

## One objective → one plan (the base model)

The base notebook minimizes cost at **full** coverage. That's a single point: the cheapest way to staff everything.

In [ ]:
base = solve(coverage_floor=TOTAL_REQUIRED)
print(f"Cheapest full-coverage plan: cover {base['coverage']}/{TOTAL_REQUIRED} shifts at ${base['cost']}  ({base['status']})")
print("That's one point. Is full coverage worth its cost vs. covering a little less? One solve can't say.")

## Two objectives, no fixed weighting → trace the frontier

Following the skill: **anchor** the objectives (coverage ranges 0…max; cost 0…full-coverage cost), then **ε-constraint sweep** — minimize cost subject to `coverage ≥ ε`, for ε across the range — and **filter** to the non-dominated set.

In [ ]:
cov_max = solve(maximize_coverage=True)["coverage"]
points = []
for eps in range(0, cov_max + 1):
    r = solve(coverage_floor=eps)
    if r:
        points.append((r["coverage"], r["cost"]))

def non_dominated(pts):                       # maximize coverage, minimize cost
    return sorted({(c, k) for (c, k) in pts
                   if not any((c2 >= c and k2 <= k and (c2 > c or k2 < k)) for (c2, k2) in pts)})

frontier = non_dominated(points)
print(f"Max achievable coverage: {cov_max}/{TOTAL_REQUIRED} | frontier points: {len(frontier)}")

In [ ]:
fr = np.array(frontier)
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(fr[:, 0], fr[:, 1], "o-", color="navy", lw=1.6, label=f"cost-vs-coverage frontier ({len(frontier)} options)")
ax.scatter([base["coverage"]], [base["cost"]], s=240, marker="*", color="crimson", zorder=5,
           label="base model: one full-coverage plan")
ax.set_xlabel("Coverage (shifts staffed)"); ax.set_ylabel("Labor cost ($)")
ax.set_title("One solve is one point; the frontier is the whole decision")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Read the frontier — this is the value

The skill's interpretation step: quote the **exchange rate** (extra $ per extra shift covered) between adjacent points, so the planner can decide *where on the curve* to sit. No single "best" — it's a choice the frontier makes visible.

In [ ]:
print("Marginal cost of coverage along the frontier:")
for i in range(1, len(frontier)):
    dcov = frontier[i][0] - frontier[i-1][0]
    dcost = frontier[i][1] - frontier[i-1][1]
    if dcov:
        print(f"  coverage {frontier[i-1][0]:2d} -> {frontier[i][0]:2d}:  +${dcost} for +{dcov} shift  (${dcost/dcov:.0f}/shift)")
print("\nThe single solve only ever showed the right-most point. The frontier shows the price of every coverage level.")

**Method note.** We use the skill's default, **ε-constraint** (minimize one objective, sweep the others as bounds): it enumerates every efficient point and stays correct when the frontier is non-convex. A weighted-sum sweep would agree on the supported points of this (convex) frontier, but on non-convex problems — common in combinatorial MILPs — it can skip efficient points entirely, which is why ε-constraint is the default.

## A second tradeoff, for free — cost vs. fairness

The base model *fixed* `max_shifts_per_worker = 4`. The skill's move — **a fixed constraint is a candidate objective** — says: sweep that cap instead of fixing it. A tighter cap spreads work more evenly (fairer) but costs more. Same ε-constraint mechanic, a different tradeoff, no new data.

In [ ]:
def solve_fairness(max_shifts, time_limit=10.0):
    prob = Problem("workforce_fairness")
    x = {p: prob.addVariable(name=f"{p[0]}_{p[1]}", vtype=VType.INTEGER, lb=0.0, ub=1.0) for p in pairs}
    obj = LinearExpression([], [], 0.0)
    for (w, s), var in x.items():
        if worker_pay[w]:
            obj += var * worker_pay[w]
    prob.setObjective(obj, sense.MINIMIZE)
    for s, req in shift_requirements.items():                 # full coverage (hard)
        e = LinearExpression([], [], 0.0); has = False
        for (w, s2), var in x.items():
            if s2 == s:
                e += var; has = True
        if has:
            prob.addConstraint(e == req, name=f"cover_{s}")
    for w in worker_pay:                                      # fairness lever: per-worker cap
        e = LinearExpression([], [], 0.0); has = False
        for (w2, s), var in x.items():
            if w2 == w:
                e += var; has = True
        if has:
            prob.addConstraint(e <= float(max_shifts), name=f"cap_{w}")
    settings = SolverSettings(); settings.set_parameter("time_limit", float(time_limit)); settings.set_parameter("log_to_console", False)
    prob.solve(settings)
    if prob.Status.name not in ("Optimal", "FeasibleFound"):
        return None
    sel = [(w, s) for (w, s), var in x.items() if var.getValue() > 0.5]
    busiest = max((sum(1 for (w2, s) in sel if w2 == w) for w in worker_pay), default=0)
    return {"max_shifts": max_shifts, "cost": sum(worker_pay[w] for (w, s) in sel), "busiest": busiest}

fair = []
for cap in range(len(shift_requirements), 0, -1):
    r = solve_fairness(cap)
    print(f"max_shifts cap {cap:2d}: " + (f"full coverage at ${r['cost']}, busiest worker {r['busiest']} shifts" if r else "INFEASIBLE (cap too tight to staff every shift)"))
    if r:
        fair.append((r["max_shifts"], r["cost"]))

if fair:
    fp = np.array(fair)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(fp[:, 0], fp[:, 1], "o-", color="seagreen", lw=1.6)
    ax.invert_xaxis()
    ax.set_xlabel("Max shifts per worker  (left = fairer)"); ax.set_ylabel("Labor cost ($) at full coverage")
    ax.set_title("cost vs. fairness: the price of spreading work evenly")
    ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Notes

- **Synthetic data** — the base notebook's toy roster; this demonstrates the *method*, not a staffing study.
- **Optimal to the gap, within the time limit** — each point is solved under a `time_limit`; points are optimal to cuOpt's gap, not certified global optima unless it returns `Optimal` at a zero gap.
- **No duals for a MILP** — an integer program has no constraint duals, so the marginal cost of coverage is read off the frontier itself (above). The continuous portfolio QP (`portfolio_optimization/QP_portfolio_frontier_duals.ipynb`) *does* expose duals — the deliberate contrast.

Built by following the `cuopt-multi-objective-exploration` skill end-to-end on cuOpt's own workforce MILP.